In [ ]:
from threading import Thread

import rclpy
from rclpy.callback_groups import ReentrantCallbackGroup
from rclpy.node import Node
from commander_py import commander
from commander_py import commander_utils
from geometry_msgs.msg import Pose, Point, Quaternion
from ur_commander.srv import VisualizePoses

rclpy.init()
node = Node("ex_pose_goal")
callback_group = ReentrantCallbackGroup()
commander = commander.Commander(
    node=node, callback_group=callback_group, move_group="ur_manipulator"
)

executor = rclpy.executors.MultiThreadedExecutor(2)
executor.add_node(node)
executor_thread = Thread(target=executor.spin, daemon=True)
executor_thread.start()
node.create_rate(1.0).sleep()

In [ ]:
def display_poses(poses: list[Pose], frmae_id: str = "base_link") -> None:

    client = node.create_client(VisualizePoses, "/commander_viz/pose_visualization")
    while not client.wait_for_service(timeout_sec=3.0):
        node.get_logger().info("Waiting for /commander_viz/pose_visualization service...")
    client.call_async(VisualizePoses.Request(poses=poses, frame_id=frmae_id))

In [ ]:
target = Pose(
    position=Point(x=0.5, y=0.0, z=0.5), orientation=Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)
)

success = display_poses([target], "base_link")
if success:
    node.get_logger().info("Pose visualization request sent successfully.")

In [ ]:
target = commander_utils.rotate_pose_euler(target, euler_deg=(0, 90, 0))
success = display_poses([target], "base_link")

In [ ]:
joint_values = [0.0, -1.57, 0.0, -1.57, 0.0, 0.0]
goal = commander.set_joint_goal(joint_values=joint_values)

In [ ]:
goal = commander.set_pose_goal(pose=target, frame_id="base_link")
print(goal)

In [ ]:
traj = commander.plan(
    pose_goal=goal,
    planner_id="",
    pipeline_id="ompl",
    acc_scale=0.2,
    vel_scale=0.2,
)